In [ ]:
import pandas as pd

In [ ]:
df = pd.read_excel('../Arkitektur Alle i Tjeneste.xlsx')
df

In [ ]:
# Add root node
groups = pd.DataFrame([{'ID': 1, 'name': 'BCC København', 'parentID': None}])

# Add L1 groups
l1_groups = df[['L1']].drop_duplicates().reset_index(drop=True)
l1_groups.rename(columns={'L1': 'name'}, inplace=True)
l1_groups['ID'] = l1_groups.index + 2
l1_groups['parentID'] = 1
groups = pd.concat([groups, l1_groups], ignore_index=True)

# Add L2 groups
l2_groups = df[['L1', 'L2']].dropna().drop_duplicates().reset_index(drop=True)
l2_groups = l2_groups.merge(groups[['ID', 'name']], left_on='L1', right_on='name', how='left')
l2_groups = l2_groups.drop(columns=['name', 'L1'])
l2_groups = l2_groups.rename(columns={'L2': 'name', 'ID': 'parentID'})
l2_groups['ID'] = l2_groups.index + 1 + groups['ID'].max()
groups = pd.concat([groups, l2_groups], ignore_index=True)

# Add L3 groups
l3_groups = df[['L2', 'L3']].dropna().drop_duplicates().reset_index(drop=True)
l3_groups = l3_groups.merge(groups[['ID', 'name']], left_on='L2', right_on='name', how='left')
l3_groups = l3_groups.drop(columns=['name', 'L2'])
l3_groups = l3_groups.rename(columns={'L3': 'name', 'ID': 'parentID'})
l3_groups['ID'] = l3_groups.index + 1 + groups['ID'].max()
groups = pd.concat([groups, l3_groups], ignore_index=True)

groups

In [ ]:
from supabase import Client, create_client
import os
from dotenv import load_dotenv
load_dotenv()

supabase: Client = create_client(
    os.environ["SUPABASE_URL"], os.environ["SUPABASE_KEY"]
)

In [ ]:

# Prepare the data for insertion
records = []
for _, row in groups.iterrows():
    records.append({
        'id': int(row['ID']),
        'name': row['name'],
        'parent_id': int(row['parentID']) if pd.notnull(row['parentID']) else None,
        # 'tenant_id': None,  # Set if needed
    })

# Insert into Supabase
response = supabase.table('groups').insert(records).execute()
print(response)